# GNN Pre-Training
Pre-trains the GNN encoder on synthetic 6G network topology data using a dummy supervised loss.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.nn.functional as F
import numpy as np
import yaml
import matplotlib.pyplot as plt
from torch_geometric.loader import DataLoader
from models.gnn import GNNModel
from utils.graph_builder import build_network_graph

In [ ]:
with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print('Config:', config)

## Generate Synthetic Dataset

In [ ]:
def generate_dataset(config, num_samples=200):
    net = config['network']
    graphs = []
    for _ in range(num_samples):
        bs_pos = np.random.uniform(0, net['area_size'], (net['num_base_stations'], 2))
        user_pos = np.random.uniform(0, net['area_size'], (net['num_users'], 2))
        g = build_network_graph(bs_pos, user_pos)
        g.y = torch.rand(g.num_nodes, 1)  # dummy regression target
        graphs.append(g)
    return graphs

dataset = generate_dataset(config)
loader = DataLoader(dataset, batch_size=16, shuffle=True)
print(f'Dataset: {len(dataset)} graphs | Batches per epoch: {len(loader)}')

## Build Model

In [ ]:
model = GNNModel(in_channels=3, hidden_channels=64, out_channels=32).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=config['training']['lr'])
print(model)

## Training Loop

In [ ]:
EPOCHS = 50
epoch_losses = []

model.train()
for epoch in range(1, EPOCHS + 1):
    total_loss = 0.0
    for batch in loader:
        batch = batch.to(device)
        out = model(batch)                          # (N, 32)
        loss = F.mse_loss(out[:, :1], batch.y)     # dummy supervised loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(loader)
    epoch_losses.append(avg_loss)
    if epoch % 10 == 0:
        print(f'Epoch {epoch:03d} | Loss: {avg_loss:.4f}')

print('Training complete.')

## Loss Curve

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(range(1, EPOCHS + 1), epoch_losses)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('GNN Pre-Training Loss')
plt.tight_layout()
plt.show()

## Save Model

In [ ]:
save_path = '../results/gnn_pretrained.pt'
os.makedirs('../results', exist_ok=True)
torch.save(model.state_dict(), save_path)
print(f'GNN saved to {save_path}')